# 09 · KG + LLM integration — realistic drug-repurposing ranking

**Task.** For a target disease, an LLM ranks a pool of candidate drugs by
repurposing plausibility. The true (post-cutoff) repurposed drug is hidden among
distractors; we score the **rank** it receives (MRR, hits@k). The only thing that
changes between arms is the knowledge-graph evidence attached to each candidate,
so any lift over the no-KG baseline is attributable to the KG.

This replaces the earlier binary yes/no plausibility design. What is new:

- **Listwise ranking**, not yes/no — closer to how a pharma analyst triages.
- **Positives from the prospective (time-split) gold standard**, so the answer is
  not already in the model's parametric memory.
- **Reason-then-JSON prompt**: free-text reasoning precedes a strict JSON block;
  the KG is attributed as *a source*, not ground truth; abstention is allowed;
  each candidate carries `basis` / `evidence_agreement` / `confidence` fields so
  we can measure whether the model *used* the KG and whether it over-trusted it.
- **Balanced dual dossiers**: each drug gets a neutral, query-independent profile
  (targets, pathways, indications, side-effects) and the disease gets one profile
  (associated genes, phenotypes). The model must connect them itself — no
  pre-computed drug→disease bridge (that would leak the answer).
- **Position-bias control**: candidate order is reshuffled across runs and the
  rank is aggregated over shuffles.
- **Model slate**: runs the same task across local Ollama models *and* hosted
  APIs (Anthropic / OpenAI / Groq / OpenRouter / Gemini).

**Modes.** `MODE = 'mock'` exercises the whole pipeline (pools → prompt → shuffles
→ parse → score) with a mock model and no KG load, so it runs anywhere. `MODE =
'real'` calls the configured models and builds KG dossiers from the real graphs.


## 1 · Setup

In [ ]:
import os, sys, json, re, string, time, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Repo root = nearest parent containing config.yaml (portable across machines).
BASE = Path.cwd().resolve()
while not (BASE / 'config.yaml').exists() and BASE != BASE.parent:
    BASE = BASE.parent
sys.path.insert(0, str(BASE / 'src'))
from loading import load_kg, load_config, find_config        # noqa: E402
from graph_utils import build_lookup_maps                    # noqa: E402
from plotting import setup_style, save_fig                   # noqa: E402

config = load_config(find_config(BASE))
GOLD   = BASE / 'data' / 'gold_standards'
FIGS   = BASE / 'results' / 'figures';                FIGS.mkdir(parents=True, exist_ok=True)
TABLES = BASE / 'results' / 'tables' / '09_llm_runs'; TABLES.mkdir(parents=True, exist_ok=True)

setup_style()
LETTERS = list(string.ascii_uppercase)
print(f'repo root: {BASE}')

## 2 · Configuration

Everything you would normally pass on the command line lives here. Start in
`'mock'` to check the plumbing, then switch to `'real'`. Hosted-API models read
their key from the matching environment variable (`ANTHROPIC_API_KEY`,
`OPENAI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`, `GEMINI_API_KEY`); local
models must be pulled in Ollama (`ollama pull <name>`).

In [ ]:
# ── Run mode ────────────────────────────────────────────────────────────────
MODE = 'mock'          # 'mock' (no Ollama/KG, runs anywhere) | 'real'
MOCK = (MODE == 'mock')
FORCE_RERUN = False    # True to ignore the cached results file and recompute

# ── Model slate (local Ollama + hosted APIs) ────────────────────────────────
# Local Ollama: a bare tag, e.g. 'llama3.3:70b' (pull it first).
# Hosted API:  'provider:model_id' for groq/openrouter/gemini, or a bare
#              'claude-*' / 'gpt-*' (provider inferred). Each needs its key in env.
MODELS = [
    'llama3.3:70b',          # local  · ollama pull llama3.3:70b
    'qwen2.5:32b',           # local  · ollama pull qwen2.5:32b
    # ── hosted APIs (uncomment + set the key, use a model id you have access to) ──
    # 'claude-sonnet-4-5',                       # ANTHROPIC_API_KEY
    # 'gpt-4o',                                  # OPENAI_API_KEY
    # 'groq:llama-3.3-70b-versatile',            # GROQ_API_KEY      (free tier)
    # 'gemini:gemini-2.0-flash',                 # GEMINI_API_KEY    (free tier)
    # 'openrouter:deepseek/deepseek-chat:free',  # OPENROUTER_API_KEY (free)
]

# ── KGs + experimental arms ─────────────────────────────────────────────────
KGS        = ['primekg']                 # KGs to attach evidence from (real mode)
CONDITIONS = ['no_kg', 'kg']             # 'no_kg' baseline runs once; 'kg' per-KG
BRIDGE_MODE = 'full'                     # 'full' = drug dossier + disease profile
                                         # 'mechanism_only' = drug dossier only
CAP         = 12                         # items/section, kept by specificity
ANONYMIZE       = False                  # code drug names, drop indications
ANONYMIZE_GENES = True                   # also code genes per-query (implies ANONYMIZE)
if ANONYMIZE_GENES:
    ANONYMIZE = True

# ── Sampling + decoding ─────────────────────────────────────────────────────
GOLD_FILE   = GOLD / 'expanded_prospective_gold_standard.tsv'
N_DISEASES  = 3
POOL_SIZE   = 8
SHUFFLES    = 3
STRATIFY    = True       # round-robin across therapeutic areas
UNIQUE_DRUG = True       # each drug is positive for at most one query
TEMPERATURE = 0.7
API_DELAY   = 1.0        # seconds before each hosted-API call (rate-limit pacing)
SEED        = 0

OUT_CSV = TABLES / '09_ranking.csv'

print(f'MODE={MODE}  models={MODELS}')
print(f'KGS={KGS}  conditions={CONDITIONS}  n_diseases={N_DISEASES}  '
      f'pool={POOL_SIZE}  shuffles={SHUFFLES}')

## 3 · Prompt (reason-then-JSON) and output schema

The prompt asks the model to weigh KG evidence against its own pharmacological
knowledge, flag conflicts, and abstain when evidence is insufficient — then emit
a single JSON object. The schema makes every reliance field required, so we can
measure how much the model leaned on the KG rather than only the final ranking.

In [ ]:
PROMPT_HEADER = (
    "You are assisting a drug repurposing scientist. For the target disease below,\n"
    "you are given candidate drugs. Some include evidence retrieved from a\n"
    "biomedical knowledge graph, reported as mechanistic links\n"
    "(drug -> target -> pathway -> disease) and related annotations.\n"
)
PROMPT_INSTRUCTIONS = (
    "Assess each candidate's mechanistic plausibility for this disease using BOTH\n"
    "the knowledge-graph evidence and your own pharmacological knowledge. The\n"
    "knowledge graph reports associations; treat them as one source, not as\n"
    "established truth. Where its evidence is sparse, or conflicts with what you\n"
    "know, say so and weight it accordingly -- do not construct a rationale to fit\n"
    "weak evidence. If the evidence is insufficient to judge a candidate, mark it\n"
    "\"uncertain\" rather than forcing a confident rank.\n\n"
    "Respond with ONLY a single JSON object (no text before or after) of the form:\n"
    "{ \"reasoning\": \"<a few sentences weighing the candidates and flagging any "
    "KG-vs-knowledge conflicts>\",\n"
    "  \"ranking\": [ { \"rank\": 1, \"drug\": \"<letter>\",\n"
    "    \"basis\": \"kg\" | \"prior\" | \"both\",\n"
    "    \"evidence_agreement\": \"consistent\" | \"conflicting\" | \"insufficient\",\n"
    "    \"confidence\": 1-5, \"rationale\": \"<one sentence>\" }, ... ] }\n"
    "Rank every candidate. ALL fields are required for EVERY candidate, even when "
    "no evidence is shown: set basis=\"prior\" when you used only your own "
    "knowledge (\"kg\"/\"both\" when you used the provided evidence), "
    "evidence_agreement=\"insufficient\" when no evidence was given; confidence "
    "(1-5) and a one-sentence rationale are ALWAYS required."
)

# JSON schema for enforced structured output (Ollama `format=`, OpenAI json_schema).
RANKING_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "ranking": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "rank": {"type": "integer"},
                    "drug": {"type": "string"},
                    "basis": {"type": "string", "enum": ["kg", "prior", "both"]},
                    "evidence_agreement": {"type": "string",
                                           "enum": ["consistent", "conflicting", "insufficient"]},
                    "confidence": {"type": "integer", "minimum": 1, "maximum": 5},
                    "rationale": {"type": "string"},
                },
                "required": ["rank", "drug", "basis", "evidence_agreement",
                             "confidence", "rationale"],
            },
        },
    },
    "required": ["reasoning", "ranking"],
}


def build_prompt(disease_name, disease_id, candidates, disease_profile=None):
    """candidates: list of dicts {letter, drug_name, evidence (str or None)}.
    disease_profile: KG disease dossier shown once in the header (KG arms only)."""
    lines = [PROMPT_HEADER, f"\nTarget disease: {disease_name} ({disease_id})"]
    if disease_profile:
        lines.append(f"Disease profile (from the knowledge graph): {disease_profile}")
    lines.append("\nCandidates (listed order is arbitrary):")
    for c in candidates:
        lines.append(f"{c['letter']}. {c['drug_name']}")
        if c['evidence']:                       # omit the line entirely if absent
            lines.append(f"   Drug profile (from the knowledge graph): {c['evidence']}")
    lines.append("")
    lines.append(PROMPT_INSTRUCTIONS)
    return "\n".join(lines)

## 4 · Response parsing

`parse_response` recovers the ranked letter order plus the per-candidate reliance
fields. If the JSON is malformed it falls back to first-mention order, and a
missing positive is scored as a miss (`rank = pool_size + 1`).

In [ ]:
def parse_ranking(text, valid_letters):
    """Ordered list of candidate letters, best first. [] if unparseable."""
    if not text:
        return []
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ranked = []
            for it in obj.get('ranking', []):
                d = str(it.get('drug', '')).strip().upper()
                d = d[0] if d and d[0] in valid_letters else None
                rk = it.get('rank')
                if d and d not in [r[0] for r in ranked]:
                    ranked.append((d, rk if isinstance(rk, (int, float)) else len(ranked) + 1))
            if ranked:
                ranked.sort(key=lambda t: t[1])
                return [d for d, _ in ranked]
        except (json.JSONDecodeError, TypeError, AttributeError):
            pass
    seen = []
    for tok in re.findall(r'\b([A-Z])\b', text.upper()):
        if tok in valid_letters and tok not in seen:
            seen.append(tok)
    return seen


def _as_int(x):
    try:
        return max(1, min(5, int(x)))
    except (ValueError, TypeError):
        return None


def parse_response(text, valid_letters):
    """Like parse_ranking, plus per-letter reliance fields
    {letter: {basis, evidence_agreement, confidence, rationale}}."""
    fields, ordered = {}, []
    if text:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                obj = json.loads(m.group(0))
                ranked = []
                for it in obj.get('ranking', []):
                    d = str(it.get('drug', '')).strip().upper()
                    d = d[0] if d and d[0] in valid_letters else None
                    if not d:
                        continue
                    if d not in fields:
                        fields[d] = {
                            'basis': (str(it.get('basis', '')).lower() or None),
                            'evidence_agreement': (str(it.get('evidence_agreement', '')).lower() or None),
                            'confidence': _as_int(it.get('confidence')),
                            'rationale': (str(it.get('rationale', '')).strip() or None),
                        }
                    if d not in [r[0] for r in ranked]:
                        rk = it.get('rank')
                        ranked.append((d, rk if isinstance(rk, (int, float)) else len(ranked) + 1))
                ranked.sort(key=lambda t: t[1])
                ordered = [d for d, _ in ranked]
            except (json.JSONDecodeError, TypeError, AttributeError):
                pass
    if not ordered:
        ordered = parse_ranking(text, valid_letters)
    return ordered, fields


def rank_of(positive_letter, ordered_letters, pool_size):
    """1-based rank of the positive; pool_size+1 (miss) if absent/unparsed."""
    if positive_letter in ordered_letters:
        return ordered_letters.index(positive_letter) + 1
    return pool_size + 1

## 5 · LLM clients — local Ollama, hosted APIs, and a mock

`rank_call` dispatches a single prompt to the right backend. Hosted models are
addressed as `provider:model_id` (or a bare `claude-*` / `gpt-*`), retry on HTTP
429 with bounded backoff, and cache whichever JSON-format flag the provider
accepts. The mock backend simulates a discerning model so the parser and scorer
are exercised without any network call.

In [ ]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
TAGS_URL   = 'http://localhost:11434/api/tags'

API_PROVIDERS = {
    'anthropic':  dict(url='https://api.anthropic.com/v1/messages',
                       key='ANTHROPIC_API_KEY', style='anthropic'),
    'openai':     dict(url='https://api.openai.com/v1/chat/completions',
                       key='OPENAI_API_KEY', style='openai'),
    'groq':       dict(url='https://api.groq.com/openai/v1/chat/completions',
                       key='GROQ_API_KEY', style='openai'),
    'openrouter': dict(url='https://openrouter.ai/api/v1/chat/completions',
                       key='OPENROUTER_API_KEY', style='openai'),
    'gemini':     dict(url='https://generativelanguage.googleapis.com/v1beta/openai/chat/completions',
                       key='GEMINI_API_KEY', style='openai'),
}
_REASONING_HINTS = ('gpt-oss', 'o1', 'o3', 'o4', '-r1', 'deepseek-r', 'qwq', 'reason', 'think')
_API_FORMAT_CACHE = {}


def _provider_model(name):
    prefix = name.split(':', 1)[0]
    if prefix in API_PROVIDERS and ':' in name:
        return prefix, name.split(':', 1)[1]
    low = name.lower()
    if low.startswith('claude'):
        return 'anthropic', name
    if low.startswith(('gpt-', 'o1', 'o3', 'o4')):
        return 'openai', name
    return None, None


def is_api_model(name):
    return _provider_model(name)[0] is not None


def _max_tokens(name):
    n = name.lower()
    return 4000 if any(h in n for h in _REASONING_HINTS) else 1500


def _api_key_for(name):
    prov, _ = _provider_model(name)
    return os.environ.get(API_PROVIDERS[prov]['key']) if prov else None


def ollama_models_available():
    import requests
    try:
        data = requests.get(TAGS_URL, timeout=3).json()
        return {m['name'] for m in data.get('models', [])}
    except Exception as e:
        raise SystemExit(f"Ollama not reachable at {TAGS_URL}: {e}")


def ollama_rank(model, prompt, seed, temperature):
    import requests
    payload = {'model': model, 'prompt': prompt, 'stream': False,
               'format': RANKING_SCHEMA,
               'options': {'temperature': temperature, 'num_predict': _max_tokens(model),
                           'num_ctx': 8192, 'seed': int(seed)}}
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=600)
        r.raise_for_status()
        return r.json().get('response', '')
    except Exception as e:
        return f'__ERROR__ {e}'


def _post_with_backoff(url, headers, body, timeout=180, max_retries=4):
    import requests
    delay = 3.0
    for _ in range(max_retries):
        try:
            r = requests.post(url, headers=headers, json=body, timeout=timeout)
        except Exception as e:
            return None, f'__ERROR__ request: {e}'
        if r.status_code == 429:
            ra = r.headers.get('retry-after', '')
            try:
                wait = float(ra)
            except ValueError:
                wait = delay
            time.sleep(min(wait, 12))
            delay = min(delay * 2, 12)
            continue
        return r, None
    return None, '__ERROR__ 429 rate-limited (gave up after retries)'


def api_rank(name, prompt, temperature):
    prov, model = _provider_model(name)
    cfg = API_PROVIDERS.get(prov)
    if not cfg:
        return '__ERROR__ not an API model'
    key = os.environ.get(cfg['key'])
    if not key:
        return f'__ERROR__ no key ({cfg["key"]})'
    mx = _max_tokens(name)
    if cfg['style'] == 'anthropic':
        r, err = _post_with_backoff(cfg['url'],
            {'x-api-key': key, 'anthropic-version': '2023-06-01', 'content-type': 'application/json'},
            {'model': model, 'max_tokens': mx, 'temperature': temperature,
             'messages': [{'role': 'user', 'content': prompt}]})
        if err:
            return err
        if r.status_code >= 400:
            return f'__ERROR__ {r.status_code} {r.text[:160]}'
        try:
            return ''.join(b.get('text', '') for b in r.json().get('content', []))
        except Exception as e:
            return f'__ERROR__ parse {e}'
    headers = {'Authorization': f'Bearer {key}', 'content-type': 'application/json'}
    base = {'model': model, 'max_tokens': mx, 'temperature': temperature,
            'messages': [{'role': 'user', 'content': prompt}]}
    ck = (prov, model)
    formats = ([_API_FORMAT_CACHE[ck]] if ck in _API_FORMAT_CACHE
               else [{'type': 'json_object'}, None])
    last = '__ERROR__ no response'
    for fmt in formats:
        body = {**base, **({'response_format': fmt} if fmt else {})}
        r, err = _post_with_backoff(cfg['url'], headers, body)
        if err:
            return err
        if r.status_code >= 400:
            last = f'__ERROR__ {r.status_code} {r.text[:160]}'
            continue
        _API_FORMAT_CACHE[ck] = fmt
        try:
            return r.json()['choices'][0]['message']['content']
        except Exception as e:
            return f'__ERROR__ parse {e}'
    return last


def mock_rank(candidates, seed):
    """Simulate a discerning model: candidates WITH evidence float up, with noise."""
    rng = np.random.default_rng(seed)
    scored = [(1.0 * bool(c['evidence']) + rng.normal(0, 0.5), c['letter']) for c in candidates]
    scored.sort(key=lambda t: -t[0])
    ordered = [l for _, l in scored]
    items = ", ".join(
        f'{{"rank": {i+1}, "drug": "{l}", "basis": "both", '
        f'"evidence_agreement": "consistent", "confidence": 3, "rationale": "mock"}}'
        for i, l in enumerate(ordered))
    return f'Reasoning: mock.\n{{"ranking": [{items}]}}'


def rank_call(model, prompt, candidates, seed):
    """Single dispatch: mock / hosted-API / local-Ollama."""
    if MOCK:
        return mock_rank(candidates, SEED + seed)
    if is_api_model(model):
        if API_DELAY:
            time.sleep(API_DELAY)
        return api_rank(model, prompt, TEMPERATURE)
    return ollama_rank(model, prompt, SEED + seed, TEMPERATURE)

## 6 · ID crosswalks and resolvers

Gold-standard drugs (DrugBank) and diseases (MONDO) have to be mapped onto each
KG's own identifier scheme before we can look them up. These crosswalks bridge
DrugBank↔external drug IDs and MONDO↔DOID↔MESH disease IDs (the same logic the
KG loaders use), and are skipped entirely in mock mode.

In [ ]:
_BARE_MESH = re.compile(r'^[DC]\d+$')


def _id_variants(cands):
    out = set(cands)
    for c in list(cands):
        if not isinstance(c, str):
            continue
        if ':' in c:
            ns, _, rest = c.partition(':')
            out.update({c.replace(':', '_', 1), f'{ns.upper()}:{rest}', f'{ns.lower()}:{rest}', rest})
            try:
                stripped = str(int(rest))
                if stripped != rest:
                    out.add(stripped)
            except (ValueError, TypeError):
                pass
        if '_' in c and ':' not in c:
            out.add(c.replace('_', ':', 1))
    return out


def load_crosswalks():
    do        = pd.read_csv(GOLD / 'do_diseases.csv')
    mesh_doid = pd.read_csv(GOLD / 'mesh_to_doid.csv', on_bad_lines='skip')
    xref      = pd.read_csv(GOLD / 'drugbank_xref.csv')
    sssom     = pd.read_csv(GOLD / 'mondo.sssom.tsv', sep='\t', comment='#')
    sssom     = sssom[sssom['predicate_id'] == 'skos:exactMatch']

    _do = do.dropna(subset=['mondo_id', 'doid']).copy()
    _do['m'] = _do['mondo_id'].str.replace('MONDO:', '', regex=False).str.lstrip('0').replace({'': '0'})
    _do['d'] = _do['doid'].str.replace('DOID:', '', regex=False).str.lstrip('0').replace({'': '0'})
    mondo_to_doid_all, doid_to_mondo_all = defaultdict(set), defaultdict(set)
    for _, r in _do.iterrows():
        mondo_to_doid_all[r['m']].add(r['d'])
        doid_to_mondo_all[r['d']].add(r['m'])

    mesh_doid['mesh_clean'] = mesh_doid['mesh_id'].str.replace(r'\s*\{.*\}', '', regex=True).str.strip()
    _mh = (mesh_doid.assign(src_rank=mesh_doid['source'].map({'DO': 0, 'MONDO': 1}).fillna(2))
                    .sort_values('src_rank').drop_duplicates('mesh_clean', keep='first'))
    mesh_to_doid = {r['mesh_clean']: (r['doid'].replace('DOID:', '').lstrip('0') or '0')
                    for _, r in _mh.iterrows() if pd.notna(r['doid'])}
    doid_to_mesh = defaultdict(set)
    for m, d in mesh_to_doid.items():
        if m.startswith('MESH:'):
            doid_to_mesh[d].add(m)

    drug_equiv = defaultdict(set)
    for db, ns, eid in zip(xref.drugbank_id.astype(str), xref.namespace.astype(str), xref.external_id.astype(str)):
        drug_equiv[db].update({f'{ns}:{eid}', eid})
        if ns == 'PUBCHEM.COMPOUND':
            drug_equiv[db].update({f'CID:{eid}', f'pubchem.compound:{eid}'})

    _eq = defaultdict(set)
    for s, o in zip(sssom['subject_id'], sssom['object_id']):
        _eq[s].add(o); _eq[o].add(s)
    disease_equiv = defaultdict(set)
    for k, vs in _eq.items():
        s = set(vs)
        for v in vs:
            s |= _eq.get(v, set())
        s.discard(k)
        disease_equiv[k] = s

    return {'mondo_to_doid_all': mondo_to_doid_all, 'doid_to_mondo_all': doid_to_mondo_all,
            'mesh_to_doid': mesh_to_doid, 'doid_to_mesh': doid_to_mesh,
            'drug_equiv': drug_equiv, 'disease_equiv': disease_equiv}


def make_resolvers(config, kg_name, xw):
    drug_equiv        = xw['drug_equiv']
    mondo_to_doid_all = xw['mondo_to_doid_all']
    doid_to_mondo_all = xw['doid_to_mondo_all']
    mesh_to_doid      = xw['mesh_to_doid']
    doid_to_mesh      = xw['doid_to_mesh']
    disease_equiv     = xw['disease_equiv']

    def to_kg_drug_ids(db_id):
        s = str(db_id).strip()
        return _id_variants({s} | drug_equiv.get(s, set()))

    def to_kg_disease_ids(disease_id):
        s = str(disease_id).strip()
        scheme = config['knowledge_graphs'][kg_name].get('disease_id_scheme', 'doid')
        if   s.startswith('MONDO:'):
            mondo = s.replace('MONDO:', '').lstrip('0') or '0'
        elif s.startswith('DOID:'):
            mondo = sorted(doid_to_mondo_all.get(s.replace('DOID:', '').lstrip('0') or '0', ['?']))[0]
        elif _BARE_MESH.match(s):
            d = mesh_to_doid.get(f'MESH:{s}')
            mondo = sorted(doid_to_mondo_all.get(d, ['?']))[0] if d else '?'
        else:
            mondo = s.lstrip('0') or '0'
        cands = set()
        if scheme == 'mondo':
            cands.update({mondo, mondo.zfill(7), f'MONDO:{mondo}'})
        elif scheme == 'doid':
            for d in mondo_to_doid_all.get(mondo, set()):
                cands.update({d, f'DOID:{d}'})
        elif scheme == 'doid_mesh':
            for d in mondo_to_doid_all.get(mondo, set()):
                cands.update({d, f'DOID:{d}'}); cands |= doid_to_mesh.get(d, set())
        elif scheme == 'mesh':
            for d in mondo_to_doid_all.get(mondo, set()):
                for m in doid_to_mesh.get(d, set()):
                    cands.update({m, m.split(':', 1)[-1]})
        for c in list(cands) + [f'MONDO:{mondo}']:
            cands |= disease_equiv.get(c, set())
        return _id_variants(cands)

    return to_kg_drug_ids, to_kg_disease_ids


def build_edge_index(kg_df):
    ix = defaultdict(list)
    for x, y, r in zip(kg_df['x_index'].values, kg_df['y_index'].values, kg_df['relation'].values):
        u, v, rs = int(x), int(y), str(r)
        ix[u].append((v, rs)); ix[v].append((u, rs))
    return ix

## 7 · KG evidence block — balanced dual dossiers

Each candidate drug gets a neutral, query-independent dossier (targets, pathways,
indications, side-effects); the disease gets one profile (associated genes,
phenotypes) shown once in the header. High-degree hubs are demoted by
*specificity* ranking (neighbours sorted by their own degree, ascending, top-`CAP`
kept) rather than a brittle threshold, and the drug's indication for the *target*
disease is masked so the answer never leaks. `make_kg_block_fn` returns one
function per dossier; it only runs in real mode.

In [ ]:
SLOT_LABEL = {
    'drug_target': 'targets', 'drug_pathway': 'modulates pathways',
    'drug_effect': 'side effects', 'drug_disease': 'approved/known for',
    'target_disease': 'associated genes', 'disease_phenotype': 'phenotypes',
}


def _load_slot_map(kg_name):
    """raw predicate -> canonical slot, from data/kg_slot_maps.yaml."""
    import yaml
    with open(BASE / 'data' / 'kg_slot_maps.yaml') as f:
        full = yaml.safe_load(f) or {}
    pred_to_slot = {}
    for slot, preds in (full.get(kg_name) or {}).items():
        for p in (preds or []):
            pred_to_slot[str(p)] = slot
    return pred_to_slot


def make_kg_block_fn(kg_name, *, bridge_mode='full', cap=12, anonymize=False, anonymize_genes=False):
    """Return (drug_dossier_fn, disease_profile_fn) for one KG. Real mode only."""
    kg_df, nodes_df = load_kg(kg_name, config)
    mp = build_lookup_maps(nodes_df)
    name_map = mp['node_name_map']
    idx_to_type = dict(zip(nodes_df['idx'].astype(int), nodes_df['type'].astype(str)))
    et = config['knowledge_graphs'][kg_name]['entity_types']
    drug_t, disease_t = et.get('Drug'), et.get('Disease')
    sep = config['knowledge_graphs'][kg_name].get('disease_id_separator')

    pred_to_slot = _load_slot_map(kg_name)
    if not pred_to_slot:
        print(f"  warning: no slot map for '{kg_name}' - KG blocks will all be empty.")

    def slot_of(rel):
        return pred_to_slot.get(str(rel))

    id_to_idx = defaultdict(list)
    for raw, idx in zip(nodes_df['id'].astype(str), nodes_df['idx'].astype(int)):
        nt = idx_to_type.get(int(idx))
        keys = {raw, raw.split('::', 1)[-1] if '::' in raw else raw}
        if sep and nt == disease_t and sep in raw:
            keys.update(p for p in raw.split(sep) if p)
        for k in keys:
            id_to_idx[k].append((int(idx), nt))

    to_drug_ids, to_disease_ids = make_resolvers(config, kg_name, xw=load_crosswalks())
    edge_index = build_edge_index(kg_df)
    degree = {u: len(adj) for u, adj in edge_index.items()}

    def _resolve(cands, want):
        for c in cands:
            for idx, t in id_to_idx.get(c, []):
                if t == want:
                    return idx
        return None

    def _neighbours_by_slot(node):
        out = defaultdict(list)
        for v, r in edge_index.get(node, []):
            s = slot_of(r)
            if s:
                out[s].append(v)
        return out

    def _topn(nodes, exclude=(), recode=None):
        seen, res = set(exclude), []
        for v in sorted(nodes, key=lambda n: degree.get(n, 0)):
            if v in seen:
                continue
            seen.add(v)
            nm = name_map.get(v, str(v))
            res.append(recode(nm) if recode else nm)
            if len(res) >= cap:
                break
        return res

    _gene_code_maps = defaultdict(dict)

    def _gene_coder(disease_id):
        m = _gene_code_maps[disease_id]
        def code(name):
            if name not in m:
                m[name] = f"Gene-{len(m) + 1}"
            return m[name]
        return code

    def drug_dossier(drug_id, drug_name, disease_id):
        d_idx = _resolve(to_drug_ids(drug_id), drug_t)
        if d_idx is None:
            return None
        by = _neighbours_by_slot(d_idx)
        recode = _gene_coder(disease_id) if anonymize_genes else None
        parts = []
        tg = _topn(by.get('drug_target', []), recode=recode)
        if tg:
            parts.append(f"{SLOT_LABEL['drug_target']}: {', '.join(tg)}")
        pw = _topn(by.get('drug_pathway', []))
        if pw:
            parts.append(f"{SLOT_LABEL['drug_pathway']}: {', '.join(pw)}")
        if not anonymize and not anonymize_genes:
            x_idx = _resolve(to_disease_ids(disease_id), disease_t)
            ind = _topn(by.get('drug_disease', []), exclude=({x_idx} if x_idx is not None else set()))
            if ind:
                parts.append(f"{SLOT_LABEL['drug_disease']}: {', '.join(ind)}")
        if not anonymize_genes:
            se = _topn(by.get('drug_effect', []))
            if se:
                parts.append(f"{SLOT_LABEL['drug_effect']}: {', '.join(se)}")
        return "; ".join(parts) if parts else None

    def disease_profile(disease_id):
        if bridge_mode == 'mechanism_only':
            return ''
        x_idx = _resolve(to_disease_ids(disease_id), disease_t)
        if x_idx is None:
            return ''
        by = _neighbours_by_slot(x_idx)
        recode = _gene_coder(disease_id) if anonymize_genes else None
        parts = []
        g = _topn(by.get('target_disease', []), recode=recode)
        if g:
            parts.append(f"{SLOT_LABEL['target_disease']}: {', '.join(g)}")
        if not anonymize_genes:
            ph = _topn(by.get('disease_phenotype', []))
            if ph:
                parts.append(f"{SLOT_LABEL['disease_phenotype']}: {', '.join(ph)}")
        return "; ".join(parts)

    return drug_dossier, disease_profile

## 8 · Candidate pools from the prospective gold standard

Each query is one disease whose true (post-cutoff) drug is hidden in a pool of
distractor drugs drawn from the rest of the gold set. Diseases are sampled
round-robin across therapeutic areas so the sample is not dominated by oncology,
and each drug is the positive for at most one query.

In [ ]:
def load_positives(gold_file):
    df = pd.read_csv(gold_file, sep='\t')
    if 'drug_id' not in df.columns and 'drugbank_id' in df.columns:
        df = df.rename(columns={'drugbank_id': 'drug_id'})
    if 'disease_id' not in df.columns and 'mondo_id' in df.columns:
        df = df.rename(columns={'mondo_id': 'disease_id'})
    need = {'drug_id', 'drug_name', 'disease_name', 'disease_id'}
    missing = need - set(df.columns)
    if missing:
        raise SystemExit(f"{gold_file.name} missing columns: {missing}")
    cols = ['drug_id', 'drug_name', 'disease_name', 'disease_id']
    out = df[cols + (['therapeutic_area'] if 'therapeutic_area' in df.columns else [])].dropna(subset=cols)
    if 'therapeutic_area' not in out.columns:
        out = out.assign(therapeutic_area='unknown')
    return out.reset_index(drop=True)


def build_queries(pos_df, n_diseases, pool_size, seed, *, unique_drug=True, stratify=True):
    rng = np.random.default_rng(seed)
    df = pos_df.copy()
    if unique_drug:
        df = df.drop_duplicates('drug_id')
    df = df.drop_duplicates('disease_name').reset_index(drop=True)
    n = min(n_diseases, len(df))
    if stratify and df['therapeutic_area'].nunique() > 1:
        areas = sorted(df['therapeutic_area'].unique())
        pools = {a: list(rng.permutation(df.index[df['therapeutic_area'] == a].to_numpy())) for a in areas}
        picks = []
        while len(picks) < n and any(pools.values()):
            for a in areas:
                if pools[a]:
                    picks.append(int(pools[a].pop()))
                    if len(picks) >= n:
                        break
        chosen = df.loc[picks]
    else:
        chosen = df.iloc[rng.choice(len(df), size=n, replace=False)]
    all_drugs = df.drop_duplicates('drug_id')[['drug_id', 'drug_name']]
    queries = []
    for _, row in chosen.iterrows():
        distractor_pool = all_drugs[all_drugs['drug_id'] != row['drug_id']]
        k = min(pool_size - 1, len(distractor_pool))
        dis = distractor_pool.iloc[rng.choice(len(distractor_pool), size=k, replace=False)]
        cands = [{'drug_id': row['drug_id'], 'drug_name': row['drug_name'], 'is_pos': True}]
        cands += [{'drug_id': d.drug_id, 'drug_name': d.drug_name, 'is_pos': False} for d in dis.itertuples()]
        queries.append({'disease_name': row['disease_name'], 'disease_id': row['disease_id'],
                        'area': row['therapeutic_area'], 'candidates': cands})
    return queries


def assign_letters_and_evidence(cands, block_for, disease_id, condition, mock, rng, code_map=None):
    """Attach a letter and (for KG conditions) an evidence string to each candidate.
    Order is shuffled here. Returns (candidate_view, positive_letter)."""
    order = list(range(len(cands)))
    rng.shuffle(order)
    view, pos_letter = [], None
    for new_i, orig_i in enumerate(order):
        c = cands[orig_i]
        letter = LETTERS[new_i]
        evidence = None
        if condition != 'no_kg':
            if mock:
                p = 0.9 if c['is_pos'] else 0.4
                evidence = "mechanistic link present" if rng.random() < p else None
            else:
                evidence = block_for(c['drug_id'], c['drug_name'], disease_id)
        if c['is_pos']:
            pos_letter = letter
        name = code_map[c['drug_id']] if code_map else c['drug_name']
        view.append({'letter': letter, 'drug_name': name, 'evidence': evidence})
    return view, pos_letter


pos_df = load_positives(GOLD_FILE)
QUERIES = build_queries(pos_df, N_DISEASES, POOL_SIZE, SEED,
                        unique_drug=UNIQUE_DRUG, stratify=STRATIFY)
print(f'{len(QUERIES)} queries  (pool={POOL_SIZE}):')
for q in QUERIES:
    print(f"  - [{q['area']}] {q['disease_name']} ({q['disease_id']})  "
          f"true drug: {q['candidates'][0]['drug_name']}")

## 9 · Ranking loop (idempotent)

For every query × condition × model × shuffle we build the prompt, call the model,
parse the ranking, and record the rank of the true drug plus the reliance fields.
The no-KG baseline runs once; each KG arm loads its graph one at a time. Results
are cached to `results/tables/09_llm_runs/09_ranking.csv` — rerun with
`FORCE_RERUN = True` to recompute.

In [ ]:
def _code_map(q):
    if not ANONYMIZE:
        return None
    ids = sorted({c['drug_id'] for c in q['candidates']})
    return {did: f"Drug-{i + 1}" for i, did in enumerate(ids)}


def run_arm(kg_tag, block_for, disease_profile_fn, conds):
    """Model loop for `conds`; rows tagged with kg_tag (or 'none' for no_kg)."""
    out, n_pos_ev, n_cells = [], 0, 0
    rl_streak = 0
    total = len(QUERIES) * len(conds) * len(MODELS) * SHUFFLES
    pbar = tqdm(total=total, desc=f'arm={kg_tag}', leave=False)
    for qi, q in enumerate(QUERIES):
        code_map = _code_map(q)
        for cond in conds:
            for model in MODELS:
                for s in range(SHUFFLES):
                    rng = np.random.default_rng(SEED + 1000 * qi + 100 * s
                                                + hash((cond, model, kg_tag)) % 97)
                    view, pos_letter = assign_letters_and_evidence(
                        q['candidates'], block_for, q['disease_id'], cond, MOCK, rng, code_map=code_map)
                    if cond != 'no_kg':
                        n_cells += 1
                        pos_ev = next(c['evidence'] for c in view if c['letter'] == pos_letter)
                        n_pos_ev += int(bool(pos_ev))
                    dprof = (disease_profile_fn(q['disease_id'])
                             if (cond != 'no_kg' and disease_profile_fn) else None)
                    prompt = build_prompt(q['disease_name'], q['disease_id'], view, disease_profile=dprof)
                    valid = [c['letter'] for c in view]
                    resp = rank_call(model, prompt, view, s + qi)
                    err = (resp[9:].strip()[:120] if isinstance(resp, str)
                           and resp.startswith('__ERROR__') else None)
                    rl_streak = rl_streak + 1 if (err and '429' in err) else 0
                    ordered, fields = parse_response(resp, valid)
                    r = rank_of(pos_letter, ordered, POOL_SIZE)
                    bvals = [v['basis'] for v in fields.values() if v.get('basis')]
                    frac_kg = (sum(b in ('kg', 'both') for b in bvals) / len(bvals) if bvals else None)
                    confs = [v['confidence'] for v in fields.values() if isinstance(v.get('confidence'), int)]
                    pf = fields.get(pos_letter, {})
                    out.append({
                        'disease': q['disease_name'], 'kg': (kg_tag if cond != 'no_kg' else 'none'),
                        'condition': cond, 'model': model, 'shuffle': s,
                        'pool_size': POOL_SIZE, 'positive_letter': pos_letter,
                        'rank': r, 'reciprocal_rank': 1.0 / r,
                        'hit@1': int(r <= 1), 'hit@3': int(r <= 3), 'hit@5': int(r <= 5),
                        'parsed': int(bool(ordered)), 'error': err,
                        'pos_basis': pf.get('basis'), 'pos_agreement': pf.get('evidence_agreement'),
                        'pos_confidence': pf.get('confidence'), 'pos_rationale': pf.get('rationale'),
                        'frac_basis_kg': frac_kg,
                        'mean_confidence': (sum(confs) / len(confs) if confs else None),
                    })
                    pbar.update(1)
                    if rl_streak >= 8:
                        pbar.close()
                        print("\nRate limit exhausted (8 calls failed in a row); aborting early "
                              "with partial results. Retry later or switch MODELS to another provider.")
                        return out, n_pos_ev, n_cells
    pbar.close()
    return out, n_pos_ev, n_cells


if OUT_CSV.exists() and not FORCE_RERUN:
    res = pd.read_csv(OUT_CSV)
    print(f'loaded cached results ({len(res)} rows) from {OUT_CSV.name}  '
          f'(set FORCE_RERUN=True to recompute)')
else:
    if not MOCK:
        ollama_wanted = [m for m in MODELS if not is_api_model(m)]
        if ollama_wanted:
            avail = ollama_models_available()
            missing = [m for m in ollama_wanted if m not in avail]
            if missing:
                print(f'  not pulled, skipping: {missing}')
                MODELS = [m for m in MODELS if is_api_model(m) or m in avail]
        for m in [m for m in MODELS if is_api_model(m)]:
            if not _api_key_for(m):
                print(f'  no API key for {m}; skipping')
                MODELS = [x for x in MODELS if x != m]
        if not MODELS:
            raise SystemExit('No usable models: pull an Ollama model or set an API key.')

    rows, coverage = [], {}
    if MOCK:
        rr, npev, ncell = run_arm('mock', None, None, CONDITIONS)
        rows += rr; coverage['mock'] = (npev, ncell)
    else:
        if 'no_kg' in CONDITIONS:
            rr, _, _ = run_arm('none', None, None, ['no_kg']); rows += rr
        if any(c != 'no_kg' for c in CONDITIONS):
            for kg in KGS:
                print(f"\nLoading KG '{kg}' (cap={CAP}, anonymize={ANONYMIZE}) ...")
                block_for, disease_profile_fn = make_kg_block_fn(
                    kg, bridge_mode=BRIDGE_MODE, cap=CAP,
                    anonymize=ANONYMIZE, anonymize_genes=ANONYMIZE_GENES)
                rr, npev, ncell = run_arm(kg, block_for, disease_profile_fn, ['kg'])
                rows += rr; coverage[kg] = (npev, ncell)
                del block_for, disease_profile_fn; gc.collect()

    res = pd.DataFrame(rows)
    res.to_csv(OUT_CSV, index=False)
    print(f'\nwrote {len(res)} rows -> {OUT_CSV}')
    if any(nc for _, nc in coverage.values()):
        print('positive coverage by KG (cells where the true drug had a dossier):')
        for kg, (npev, ncell) in coverage.items():
            if ncell:
                print(f'  {kg:<12} {npev}/{ncell}')

## 10 · Results — MRR / hits@k per model × KG × condition

Higher MRR and hits@k mean the true drug is ranked nearer the top. `reliance` is
the mean fraction of candidates the model said it judged using KG evidence
(`basis` ∈ {kg, both}); `conf` is mean self-reported confidence; `parse_rate` is
the fraction of responses that yielded a valid ranking.

In [ ]:
summary = (res.groupby(['model', 'kg', 'condition'])
           .agg(MRR=('reciprocal_rank', 'mean'),
                hit1=('hit@1', 'mean'), hit3=('hit@3', 'mean'), hit5=('hit@5', 'mean'),
                reliance=('frac_basis_kg', 'mean'), conf=('mean_confidence', 'mean'),
                parse_rate=('parsed', 'mean'), n=('rank', 'size'))
           .round(3).reset_index().sort_values(['model', 'condition', 'kg']))
if 'error' in res and res['error'].notna().any():
    n_err = int(res['error'].notna().sum())
    print(f'note: {n_err}/{len(res)} calls errored (counted as misses, they deflate MRR)')
summary

## 11 · Figure — does KG context lift the ranking?

Mean MRR per model, no-KG baseline vs KG arm (averaged over KGs), with 95% CIs
across queries × shuffles. A positive gap means the knowledge-graph evidence
helped the model rank the true repurposing candidate higher.

In [ ]:
def _ci95(x):
    x = np.asarray(x, float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0

agg = (res.groupby(['model', 'condition'])['reciprocal_rank']
       .agg(['mean', _ci95]).rename(columns={'_ci95': 'ci'}).reset_index())
models = list(dict.fromkeys(agg['model']))
conds  = [c for c in ['no_kg', 'kg'] if c in set(agg['condition'])]
x = np.arange(len(models)); w = 0.8 / max(len(conds), 1)
fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(models)), 4.2))
for j, cond in enumerate(conds):
    sub = agg[agg['condition'] == cond].set_index('model').reindex(models)
    ax.bar(x + j * w, sub['mean'].values, w, yerr=sub['ci'].values, capsize=3,
           label=('no KG' if cond == 'no_kg' else 'with KG'))
ax.axhline(np.mean([1.0 / r for r in range(1, POOL_SIZE + 1)]) / 1, ls=':', c='gray', lw=1,
           label='_nolegend_')
ax.set_xticks(x + w * (len(conds) - 1) / 2)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Mean reciprocal rank (MRR)')
ax.set_title('Effect of KG context on LLM repurposing ranking')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, FIGS, '09_ranking_mrr')
plt.show()